# MoviBogotá S.A. · Entrega 3 — Modelo predictivo de demanda por estación

**Pregunta BA (P3):** ¿Cuántas validaciones se esperan por estación en cada mes del próximo trimestre?

**Decisión que apoya:** cómo repartir la flota y las frecuencias entre troncales y estaciones para el trimestre siguiente.

- Variable objetivo: `validaciones` = suma de `fact_validaciones.cantidad_validaciones` por estación y mes.
- Tipo de problema: **regresión**.
- Criterio de éxito: **MAE menor a 5 validaciones por estación y mes** en el periodo de prueba (oct–dic 2025) y menor que el del pronóstico ingenuo (repetir el valor de hace 3 meses).
- Regla de elección entre algoritmos: menor MAE; si difieren en menos de 0,1 validaciones, decide el menor RMSE.

El notebook corre de principio a fin: extrae los datos del DW con `extraccion.sql`, explora, prepara, entrena dos algoritmos, los evalúa y escribe las predicciones en `dw.pred_validaciones_estacion_mes`.

## 0. Configuración y conexión al data warehouse

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# La cadena de conexión NO se guarda en el repositorio: se toma de la variable de entorno DW_URL.
# Ejemplo local: postgresql+psycopg2://postgres:<clave>@localhost:5432/movibogota_dw
DW_URL = os.environ.get("DW_URL", "postgresql+psycopg2://postgres@localhost:5432/movibogota_dw")
engine = create_engine(DW_URL)

BASE = Path.cwd() if (Path.cwd() / "extraccion.sql").exists() else Path.cwd() / "modelo"
FIG = BASE / "figuras"
FIG.mkdir(exist_ok=True)
SEMILLA = 42
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Carpeta del modelo:", BASE)

Carpeta del modelo: /home/claude/repo/modelo


## 1. Extracción del conjunto de datos (`extraccion.sql`)

In [2]:
consulta = (BASE / "extraccion.sql").read_text(encoding="utf-8")
with engine.connect() as con:
    df = pd.read_sql(text(consulta), con)
df["fecha_mes"] = pd.to_datetime(dict(year=df.anio, month=df.mes, day=1))
print(f"Registros: {len(df):,}  ·  Estaciones: {df.sk_estacion.nunique()}  ·  "
      f"Periodo: {df.fecha_mes.min():%Y-%m} a {df.fecha_mes.max():%Y-%m}")
df.head()

Registros: 2,544  ·  Estaciones: 106  ·  Periodo: 2024-01 a 2025-12


,sk_estacion,sk_tiempo,cod_estacion,nombre_estacion,nombre_troncal,tipo_estacion,num_vagones,num_accesos,area_m2,anio,mes,dias_mes,dias_habiles,validaciones,validaciones_pico,fecha_mes
0,2,20240101,06103,CAN,Calle 26,Sencilla,2,1,289.44,2024,1,31,23,14,10,2024-01-01
1,2,20240201,06103,CAN,Calle 26,Sencilla,2,1,289.44,2024,2,29,21,26,15,2024-02-01
2,2,20240301,06103,CAN,Calle 26,Sencilla,2,1,289.44,2024,3,31,21,16,9,2024-03-01
3,2,20240401,06103,CAN,Calle 26,Sencilla,2,1,289.44,2024,4,30,22,13,4,2024-04-01
4,2,20240501,06103,CAN,Calle 26,Sencilla,2,1,289.44,2024,5,31,23,22,12,2024-05-01


In [3]:
df.dtypes.to_frame("tipo")

,tipo
sk_estacion,int64
sk_tiempo,int64
cod_estacion,str
nombre_estacion,str
nombre_troncal,str
tipo_estacion,str
num_vagones,int64
num_accesos,int64
area_m2,float64
anio,int64


## 2. Comprensión de los datos

### 2.1 Distribución de la variable objetivo

In [4]:
y = df["validaciones"]
q1, q3 = y.quantile([0.25, 0.75])
lim_sup = q3 + 1.5 * (q3 - q1)
atipicos = df[y > lim_sup]
print(y.describe().round(2))
print(f"Meses en 0: {(y == 0).sum()}  ·  Límite IQR de atípicos: {lim_sup:.1f}  ·  "
      f"Registros atípicos: {len(atipicos)} ({len(atipicos)/len(df):.1%})")
print("Estaciones con atípicos:", atipicos.nombre_estacion.value_counts().head(8).to_dict())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [3, 1]})
ax[0].hist(y, bins=40, color="#3b6fd8", edgecolor="white")
ax[0].axvline(y.mean(), color="#c0392b", ls="--", label=f"media {y.mean():.1f}")
ax[0].axvline(y.median(), color="#2c3e50", ls=":", label=f"mediana {y.median():.0f}")
ax[0].set(title="Validaciones por estación y mes", xlabel="validaciones", ylabel="n.º de estación-mes")
ax[0].legend()
ax[1].boxplot(y, orientation="vertical", widths=0.5)
ax[1].set(title="Diagrama de caja", ylabel="validaciones", xticks=[])
fig.tight_layout(); fig.savefig(FIG / "01_distribucion_objetivo.png", bbox_inches="tight"); plt.show()

count    2544.00
mean       21.52
std        13.25
min         4.00
25%        14.00
50%        18.00
75%        25.00
max       120.00
Name: validaciones, dtype: float64
Meses en 0: 0  ·  Límite IQR de atípicos: 41.5  ·  Registros atípicos: 169 (6.6%)
Estaciones con atípicos: {'Portal Norte - Unicervantes': 24, 'Portal El Dorado - C.C. Nuestro Bogotá': 24, 'Portal Américas': 24, 'Portal Sur - JFK Coop. Financiera': 24, 'Portal 20 de Julio': 23, 'Portal Tunal': 21, 'Portal Usme': 19, 'Banderas': 3}


### 2.2 Análisis exploratorio: relación con la variable objetivo

In [5]:
# Gráfico 1: tendencia y estacionalidad (total de la red por mes)
red = df.groupby("fecha_mes", as_index=False)["validaciones"].sum()
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(red.fecha_mes, red.validaciones, marker="o", color="#3b6fd8")
for x, v in zip(red.fecha_mes, red.validaciones):
    if x.month in (6, 12):
        ax.annotate(f"{v:,}".replace(",", "."), (x, v), textcoords="offset points", xytext=(0, -14),
                    ha="center", fontsize=8, color="#c0392b")
ax.set(title="Validaciones de la red por mes, 2024–2025", ylabel="validaciones")
fig.tight_layout(); fig.savefig(FIG / "02_tendencia_mensual.png", bbox_inches="tight"); plt.show()
print(red.assign(anio=red.fecha_mes.dt.year).groupby("anio").validaciones.sum())

anio
2024    26467
2025    28270
Name: validaciones, dtype: int64


In [6]:
# Gráfico 2: tipo de estación
orden = df.groupby("tipo_estacion").validaciones.median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.boxplot([df.loc[df.tipo_estacion == t, "validaciones"] for t in orden], tick_labels=orden)
ax.set(title="Validaciones por estación y mes según tipo de estación", ylabel="validaciones")
fig.tight_layout(); fig.savefig(FIG / "03_tipo_estacion.png", bbox_inches="tight"); plt.show()
df.groupby("tipo_estacion").validaciones.agg(["count", "mean", "median", "max"]).round(1).loc[orden]

,count,mean,median,max
tipo_estacion,,,,
Portal,168,60.8,58.0,120
Intermedia,168,28.8,29.0,52
Temporal,168,28.9,29.0,45
Sencilla,2040,17.1,16.0,42


In [7]:
# Gráfico 3: el pasado de la misma estación (valor de hace 3 meses) frente al mes actual
tmp = df.sort_values(["sk_estacion", "fecha_mes"]).copy()
tmp["hace_3_meses"] = tmp.groupby("sk_estacion").validaciones.shift(3)
tmp = tmp.dropna(subset=["hace_3_meses"])
r = tmp[["hace_3_meses", "validaciones"]].corr().iloc[0, 1]
fig, ax = plt.subplots(figsize=(5.5, 4.4))
ax.scatter(tmp.hace_3_meses, tmp.validaciones, s=8, alpha=0.35, color="#3b6fd8")
lim = tmp[["hace_3_meses", "validaciones"]].max().max()
ax.plot([0, lim], [0, lim], color="#c0392b", lw=1, ls="--", label="y = x")
ax.set(title=f"Mes actual vs. hace 3 meses (r = {r:.2f})", xlabel="validaciones hace 3 meses",
       ylabel="validaciones del mes")
ax.legend(); fig.tight_layout(); fig.savefig(FIG / "04_rezago_3_meses.png", bbox_inches="tight"); plt.show()
print(f"Correlación de Pearson: {r:.3f}")

Correlación de Pearson: 0.850


In [8]:
# Gráfico 4: troncal de la estación
por_troncal = df.groupby("nombre_troncal").validaciones.mean().sort_values()
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(por_troncal.index, por_troncal.values, color="#3b6fd8")
ax.set(title="Promedio de validaciones por estación y mes, por troncal", xlabel="validaciones")
fig.tight_layout(); fig.savefig(FIG / "05_troncal.png", bbox_inches="tight"); plt.show()
por_troncal.round(1).sort_values(ascending=False)

nombre_troncal
Caracas            29.1
Autopista Norte    24.9
Carrera 10         21.0
Américas           20.1
Caracas Sur        19.1
Calle 26           18.7
NQS Sur            18.4
Name: validaciones, dtype: float64

## 3. Preparación de los datos

### 3.1 Nulos, atípicos y filas del horizonte de pronóstico

In [9]:
nulos = df.isna().sum()
print("Nulos por columna:\n", nulos[nulos > 0])
n_area_nula = int(df.area_m2.isna().sum())
est_area_nula = df.loc[df.area_m2.isna(), "sk_estacion"].nunique()
# area_m2: se imputa con la mediana de su tipo de estación (las estaciones sin área son pocas y del mismo tipo)
df["area_m2"] = df.groupby("tipo_estacion").area_m2.transform(lambda s: s.fillna(s.median()))
df["area_m2"] = df.area_m2.fillna(df.area_m2.median())
print(f"area_m2 imputada en {n_area_nula} registros ({est_area_nula} estaciones)")
# Atípicos: se conservan (son meses reales de los portales de mayor demanda; recortarlos sesgaría el pronóstico)
print(f"Atípicos por IQR (> {lim_sup:.1f}): {len(atipicos)} registros, conservados")

# Filas del trimestre a pronosticar (ene–mar 2026): mismas estaciones, objetivo desconocido
fut = []
for f in pd.date_range("2026-01-01", "2026-03-01", freq="MS"):
    fin = f + pd.offsets.MonthEnd(0)
    base = df.drop_duplicates("sk_estacion").copy()
    base["fecha_mes"], base["anio"], base["mes"] = f, f.year, f.month
    base["sk_tiempo"] = int(f.strftime("%Y%m%d"))
    base["dias_mes"] = fin.day
    base["dias_habiles"] = int(np.busday_count(f.date(), (fin + pd.Timedelta(days=1)).date()))
    base["validaciones"] = np.nan
    base["validaciones_pico"] = np.nan
    fut.append(base)
panel = pd.concat([df, *fut], ignore_index=True).sort_values(["sk_estacion", "fecha_mes"]).reset_index(drop=True)
print("Panel con horizonte:", panel.shape)

Nulos por columna:
 area_m2    264
dtype: int64
area_m2 imputada en 264 registros (11 estaciones)
Atípicos por IQR (> 41.5): 169 registros, conservados
Panel con horizonte: (2862, 16)


### 3.2 Variables derivadas

Todas usan solo información de **3 o más meses antes** del mes que se predice: al empezar un trimestre ya se conocen para sus tres meses.

In [10]:
g = panel.groupby("sk_estacion")["validaciones"]
panel["lag_3"] = g.shift(3)
panel["lag_4"] = g.shift(4)
panel["lag_5"] = g.shift(5)
panel["media_lag_3a5"] = panel[["lag_3", "lag_4", "lag_5"]].mean(axis=1)
panel["media_historica"] = g.transform(lambda s: s.shift(3).expanding().mean())
panel["participacion_pico_hist"] = (
    panel.groupby("sk_estacion")["validaciones_pico"].transform(lambda s: s.shift(3).expanding().sum())
    / panel.groupby("sk_estacion")["validaciones"].transform(lambda s: s.shift(3).expanding().sum())
).fillna(0)

NUM = ["lag_3", "lag_4", "lag_5", "media_lag_3a5", "media_historica", "participacion_pico_hist",
       "mes", "dias_habiles", "num_vagones", "num_accesos", "area_m2"]
CAT = ["tipo_estacion", "nombre_troncal"]
OBJ = "validaciones"
panel[NUM + CAT].describe(include="all").T[["count", "mean", "min", "max"]]

,count,mean,min,max
lag_3,2544.0,21.516116,4.0,120.0
lag_4,2438.0,21.606235,4.0,120.0
lag_5,2332.0,21.536878,4.0,120.0
media_lag_3a5,2544.0,21.439007,5.0,108.0
media_historica,2544.0,20.762349,5.0,86.26087
participacion_pico_hist,2862.0,0.504142,0.0,0.916667
mes,2862.0,6.0,1.0,12.0
dias_habiles,2862.0,21.740741,20.0,23.0
num_vagones,2862.0,2.320755,0.0,6.0
num_accesos,2862.0,1.462264,1.0,3.0


### 3.3 División temporal en entrenamiento y prueba

In [11]:
# Las primeras 5 filas de cada estación (ene–may 2024) no tienen lag_5 y se excluyen.
modelo_df = panel.dropna(subset=["lag_5"])
train = modelo_df[(modelo_df.fecha_mes >= "2024-06-01") & (modelo_df.fecha_mes <= "2025-09-01")]
test = modelo_df[(modelo_df.fecha_mes >= "2025-10-01") & (modelo_df.fecha_mes <= "2025-12-01")]
horizonte = modelo_df[modelo_df.fecha_mes >= "2026-01-01"]
for nombre, d in [("Entrenamiento", train), ("Prueba", test), ("Pronóstico", horizonte)]:
    print(f"{nombre:13s} {len(d):5,} registros  {d.fecha_mes.min():%Y-%m} a {d.fecha_mes.max():%Y-%m}")
print(f"Proporción prueba / (entrenamiento + prueba): {len(test)/(len(train)+len(test)):.1%}")
X_train, y_train = train[NUM + CAT], train[OBJ]
X_test, y_test = test[NUM + CAT], test[OBJ]

Entrenamiento 1,696 registros  2024-06 a 2025-09
Prueba          318 registros  2025-10 a 2025-12
Pronóstico      318 registros  2026-01 a 2026-03
Proporción prueba / (entrenamiento + prueba): 15.8%


## 4. Modelado: dos algoritmos con el mismo conjunto y las mismas variables

In [12]:
def preprocesador():
    return ColumnTransformer([
        ("num", StandardScaler(), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
    ])

modelos = {
    "Regresión lineal": Pipeline([("prep", preprocesador()), ("modelo", LinearRegression())]),
    "Bosque aleatorio": Pipeline([("prep", preprocesador()),
                                  ("modelo", RandomForestRegressor(n_estimators=300, max_depth=8,
                                                                   min_samples_leaf=5, n_jobs=-1,
                                                                   random_state=SEMILLA))]),
}
for nombre, m in modelos.items():
    m.fit(X_train, y_train)
    print("Entrenado:", nombre, "·", m.named_steps["modelo"])

Entrenado: Regresión lineal · LinearRegression()


Entrenado: Bosque aleatorio · RandomForestRegressor(max_depth=8, min_samples_leaf=5, n_estimators=300,
                      n_jobs=-1, random_state=42)


## 5. Evaluación sobre el conjunto de prueba (oct–dic 2025)

In [13]:
def metricas(y_real, y_pred):
    return mean_absolute_error(y_real, y_pred), mean_squared_error(y_real, y_pred) ** 0.5

pred = {n: np.clip(m.predict(X_test), 0, None) for n, m in modelos.items()}
pred["Pronóstico ingenuo (lag_3)"] = test["lag_3"].to_numpy()
filas = []
for n, p in pred.items():
    mae, rmse = metricas(y_test, p)
    total_real, total_pred = y_test.sum(), p.sum()
    filas.append({"algoritmo": n, "MAE": round(mae, 2), "RMSE": round(rmse, 2),
                  "error_total_trimestre_%": round((total_pred - total_real) / total_real * 100, 1)})
resultados = pd.DataFrame(filas).set_index("algoritmo")
print(f"Media real de prueba: {y_test.mean():.2f} validaciones por estación y mes · total real {y_test.sum():,.0f}")
resultados

Media real de prueba: 22.78 validaciones por estación y mes · total real 7,243


,MAE,RMSE,error_total_trimestre_%
algoritmo,,,
Regresión lineal,4.29,5.90,-3.6
Bosque aleatorio,4.28,6.01,-6.4
Pronóstico ingenuo (lag_3),6.10,8.05,1.1


In [14]:
# Regla de elección (fijada antes de ver la prueba): gana el menor MAE; si los dos MAE difieren en
# menos de 0,1 validaciones (empate práctico), decide el menor RMSE (castiga más los errores grandes).
cand = resultados.drop(index="Pronóstico ingenuo (lag_3)").sort_values("MAE")
if cand.MAE.iloc[1] - cand.MAE.iloc[0] < 0.1:
    mejor = cand.RMSE.idxmin()
    print(f"Empate práctico en MAE ({cand.MAE.iloc[0]} vs {cand.MAE.iloc[1]}): decide el RMSE")
else:
    mejor = cand.index[0]
print("Algoritmo elegido:", mejor)
mae_mejor = resultados.loc[mejor, "MAE"]
mae_ingenuo = resultados.loc["Pronóstico ingenuo (lag_3)", "MAE"]
print(f"Criterio de éxito: MAE < 5 y menor que el ingenuo ({mae_ingenuo})  →  "
      f"MAE = {mae_mejor}  →  {'SÍ' if (mae_mejor < 5 and mae_mejor < mae_ingenuo) else 'NO'} se alcanza")

# Error por mes de la red y por estación grande
t = test.assign(pred=pred[mejor], err=lambda d: (d.pred - d.validaciones).abs())
print(t.groupby("fecha_mes").agg(real=("validaciones", "sum"), predicho=("pred", "sum"), MAE=("err", "mean")).round(1))
print(t.groupby("tipo_estacion").err.mean().round(2))

Empate práctico en MAE (4.28 vs 4.29): decide el RMSE
Algoritmo elegido: Regresión lineal
Criterio de éxito: MAE < 5 y menor que el ingenuo (6.1)  →  MAE = 4.29  →  SÍ se alcanza
              real  predicho  MAE
fecha_mes                        
2025-10-01  2730.0    2363.6  4.7
2025-11-01  2452.0    2261.6  4.0
2025-12-01  2061.0    2360.0  4.2
tipo_estacion
Intermedia     3.97
Portal        10.34
Sencilla       3.64
Temporal       6.46
Name: err, dtype: float64


In [15]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharex=True, sharey=True)
for ax, n in zip(axes, ["Regresión lineal", "Bosque aleatorio"]):
    ax.scatter(y_test, pred[n], s=10, alpha=0.5, color="#3b6fd8")
    lim = max(y_test.max(), pred[n].max()) * 1.05
    ax.plot([0, lim], [0, lim], ls="--", color="#c0392b", lw=1)
    ax.set(title=f"{n} · MAE {resultados.loc[n,'MAE']:.2f}", xlabel="real", ylabel="predicho")
fig.suptitle("Real vs. predicho por estación y mes (prueba oct–dic 2025)")
fig.tight_layout(); fig.savefig(FIG / "06_real_vs_predicho.png", bbox_inches="tight"); plt.show()

### 5.3 Variables más importantes del modelo elegido

In [16]:
m = modelos[mejor]
nombres = m.named_steps["prep"].get_feature_names_out()
nombres = [s.split("__", 1)[1] for s in nombres]
if hasattr(m.named_steps["modelo"], "feature_importances_"):
    imp = pd.Series(m.named_steps["modelo"].feature_importances_, index=nombres)
    etiqueta = "importancia (reducción de error)"
else:
    imp = pd.Series(m.named_steps["modelo"].coef_, index=nombres)
    etiqueta = "coeficiente (validaciones por 1 desviación estándar o por categoría)"
imp = imp.reindex(imp.abs().sort_values(ascending=False).index)
top = imp.head(10)[::-1]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.barh(top.index, top.values, color=["#3b6fd8" if v >= 0 else "#c0392b" for v in top.values])
ax.axvline(0, color="#555", lw=0.8)
ax.set(title=f"Variables más importantes · {mejor}", xlabel=etiqueta)
fig.tight_layout(); fig.savefig(FIG / "07_importancia_variables.png", bbox_inches="tight"); plt.show()
imp.head(10).round(2)

tipo_estacion_Portal              11.82
media_historica                    8.90
tipo_estacion_Temporal            -6.44
tipo_estacion_Sencilla            -5.43
nombre_troncal_Caracas             4.69
nombre_troncal_Caracas Sur        -2.08
nombre_troncal_Carrera 10         -1.59
nombre_troncal_Autopista Norte     1.57
nombre_troncal_NQS Sur            -1.25
lag_3                             -1.03
dtype: float64

### 5.5 Respuesta a la pregunta: pronóstico del primer trimestre de 2026

In [17]:
# Para pronosticar se reentrena el algoritmo elegido con todo el periodo conocido (jun 2024 – dic 2025),
# con los mismos parámetros y variables.
from sklearn.base import clone
final = clone(modelos[mejor]).fit(pd.concat([X_train, X_test]), pd.concat([y_train, y_test]))
horizonte = horizonte.assign(pred=np.clip(final.predict(horizonte[NUM + CAT]), 0, None))
q1_2025 = df[(df.anio == 2025) & (df.mes <= 3)].validaciones.sum()
total = horizonte.pred.sum()
print(f"Pronóstico ene–mar 2026: {total:,.0f} validaciones (ene–mar 2025 real: {q1_2025:,}; "
      f"variación {(total/q1_2025-1)*100:+.1f} %)")
print(horizonte.groupby("fecha_mes").pred.sum().round(0))
por_troncal = (horizonte.groupby("nombre_troncal").pred.sum().sort_values(ascending=False).to_frame("pronostico"))
por_troncal["participacion_%"] = (por_troncal.pronostico / total * 100).round(1)
por_troncal["buses_si_flota_proporcional"] = (por_troncal.pronostico / total * 450).round(0).astype(int)
print(por_troncal.round(0))
top_est = horizonte.groupby("nombre_estacion").pred.sum().sort_values(ascending=False).head(10).round(0)
print(top_est)

Pronóstico ene–mar 2026: 6,894 validaciones (ene–mar 2025 real: 6,679; variación +3.2 %)
fecha_mes
2026-01-01    2269.0
2026-02-01    2264.0
2026-03-01    2361.0
Name: pred, dtype: float64
                 pronostico  participacion_%  buses_si_flota_proporcional
nombre_troncal                                                           
Autopista Norte      1275.0             18.0                           83
Caracas              1240.0             18.0                           81
Américas             1038.0             15.0                           68
Caracas Sur           982.0             14.0                           64
NQS Sur               943.0             14.0                           62
Calle 26              778.0             11.0                           51
Carrera 10            638.0              9.0                           42
nombre_estacion
Portal Norte - Unicervantes               241.0
Portal Américas                           201.0
Portal El Dorado - C.C. Nuestro B

## 6. Despliegue: predicciones en `dw.pred_validaciones_estacion_mes`

In [18]:
ddl = (BASE.parent / "dw" / "ddl" / "10_pred_validaciones_estacion_mes.sql").read_text(encoding="utf-8")
salida = pd.concat([
    t.assign(tipo_registro="prueba", validaciones_reales=t.validaciones.astype(int),
             error_absoluto=t.err.round(2)),
    horizonte.assign(tipo_registro="pronostico", validaciones_reales=pd.NA, error_absoluto=np.nan),
])[["sk_estacion", "sk_tiempo", "anio", "mes", "tipo_registro", "validaciones_reales", "pred", "error_absoluto"]]
salida = salida.rename(columns={"pred": "validaciones_predichas"})
salida["validaciones_predichas"] = salida.validaciones_predichas.round(2)
salida["modelo"] = mejor
with engine.begin() as con:
    con.execute(text(ddl))
    con.execute(text("DELETE FROM dw.pred_validaciones_estacion_mes"))
    salida.to_sql("pred_validaciones_estacion_mes", con, schema="dw", if_exists="append", index=False)
    n = con.execute(text("SELECT tipo_registro, COUNT(*) FROM dw.pred_validaciones_estacion_mes GROUP BY 1 ORDER BY 1")).all()
print("Filas escritas:", n)
salida.to_csv(BASE / "predicciones.csv", index=False)
salida.head()

Filas escritas: [('pronostico', 318), ('prueba', 318)]


,sk_estacion,sk_tiempo,anio,mes,tipo_registro,validaciones_reales,validaciones_predichas,error_absoluto,modelo
21,2,20251001,2025,10,prueba,17,16.73,0.27,Regresión lineal
22,2,20251101,2025,11,prueba,13,16.13,3.13,Regresión lineal
23,2,20251201,2025,12,prueba,17,16.98,0.02,Regresión lineal
48,4,20251001,2025,10,prueba,22,14.11,7.89,Regresión lineal
49,4,20251101,2025,11,prueba,12,13.24,1.24,Regresión lineal
